In [9]:
! ls /opt/workspace/

chrome-extensions-samples  dify			  htm	       py
core-java-13th		   geogebra		  javaworks
corejava_13th		   google-chrome-samples  module-test


In [11]:
! date
! pip install --upgrade pip
! pip install torch torchvision transformer_utils tqdm

Wed Feb 18 05:43:09 PM CST 2026
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 1.8 MB/s  0:00:01 eta 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [2]:

#1.radeontop
#来查看 AMD 显卡的实时占用
#2.rocm-smi
#ROCm 自带的监控工具，类似 NVIDIA 的 nvidia-smi

import os
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "11.0.2"
import torch
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from torch.autograd import variable


# 检查CUDA是否可用
if torch.cuda.is_available():
    print("CUDA is available. GPU is ON!")
    print("CUDA version:", torch.version.cuda)
    print("Number of GPUs available:", torch.cuda.device_count())
else:
    print("CUDA is not available. GPU is OFF.")


device = torch.device('cuda:0') # 选择第一个GPU
print(f"device:{device}")

# 获取第一个GPU的属性
device_props = torch.cuda.get_device_properties(0)
print(f"Total GPU memory: {device_props.total_memory / 1024**2} MB")

print(f"GPU 可用: {torch.cuda.is_available()}")
print(f"cuda 是否构建: {torch.backends.cuda.is_built()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")
print(f"current device: {torch.cuda.current_device()}")


print(f"torch version: {torch.__version__}")
print(f"当前rocm设置: {os.environ.get('HSA_OVERRIDE_GFX_VERSION')}")


CUDA is available. GPU is ON!
CUDA version: None
Number of GPUs available: 1
device:cuda:0
Total GPU memory: 6876.94140625 MB
GPU 可用: True
cuda 是否构建: True
GPU name: AMD Radeon 780M Graphics
current device: 0
torch version: 2.10.0+rocm7.1
当前rocm设置: 11.0.2


In [3]:
# 全部元素是1 的二维张量
a = torch.ones(3, 3).to('cuda')
b = torch.ones(3, 3).to('cuda')
#张量加
print(a + b)

#张量加, 再乘标量
print((a + b) * 1/10)



tensor([[2., 2., 2.],
        [2., 2., 2.],
        [2., 2., 2.]], device='cuda:0')
tensor([[0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000]], device='cuda:0')


In [ ]:
# print(dir(torch.get_default_dtype))
# print(dir(torch.float8_e4m3fn))
zeros = torch.zeros(size=(2, 2))
print(zeros)

a = torch.ones(3, 3).to("cuda")
b = torch.ones(3, 3).to("cuda")
addResult = a + b
#张量加
print(f"a + b:\n{addResult}")
print(f"add.shape: {addResult.shape}, {addResult.dtype}, {addResult.device}")
#张量加, 再乘标量
print(f"(a + b) * 1/10:\n{addResult * 1/10}")

a1 = torch.ones(3, 5).to("cuda") * 2
b1 = torch.ones(3, 5).to("cuda") * 3
#张量乘，注意维度要一致
aa = a1 * b1
print(f"a1 * b1:\n{aa}")
# 重塑（reshape）为长度为 15 的一维视图
# 元素总数不变：新形状的元素总数必须与原张量的元素总数完全一致。
# 共享数据内存：view 返回的新张量与原张量共享底层数据。
print(f"aa.view(1, 15):\n{aa.view(aa.numel())}")

# 当张量经过转置 (transpose) 或置换 (permute) 后，内存可能不连续。
# 此时 view 会失败，而 reshape 会自动处理（必要时复制数据）。
print(aa.reshape(-1))

print(torch.rand(size=(3,6), dtype=torch.float16))
# RuntimeError: "check_uniform_bounds" not implemented for 'float8_e4m3fn'
# print(torch.rand(4, 4, dtype=torch.float8_e4m3fn))

#FP8格式有两种变体 E4M3(4位指数和3位尾数)和E5M2(5位指数和2位尾数)
print(torch.empty(4, 7, dtype=torch.float8_e4m3fn))

vector = torch.tensor([7, 7])
matrix  = torch.tensor([[7, 8], 
                       [9, 10]])

zero_to_ten = torch.arange(start=0, end=10, step=1)
print(f"zero_to_ten: {zero_to_ten}")
ten_zeros = torch.zeros_like(input=zero_to_ten) # will have same shape
print(f"ten_zeros: {ten_zeros}")

ten_zeros = torch.ones_like(input=zero_to_ten, dtype=torch.float8_e4m3fn).to("cuda") # will have same shape
print(f"ten_zeros: {ten_zeros}")


# float16: 牺牲了范围，换取了更高的精度（更多小数位）。
# bfloat16: 牺牲了精度，换取了与 float32 相同的动态范围
# 获取 float16 的信息
finfo = torch.finfo(torch.float16)

print(f"数据类型: {finfo.dtype}")
print(f"最大值: {finfo.max}")
print(f"最小值: {finfo.min}")
print(f"最小正规正数: {finfo.tiny}")
print(f"精度 (eps): {finfo.eps}")


tensor([[0., 0.],
        [0., 0.]])
a + b:
tensor([[2., 2., 2.],
        [2., 2., 2.],
        [2., 2., 2.]], device='cuda:0')
add.shape: torch.Size([3, 3]), torch.float32, cuda:0
(a + b) * 1/10:
tensor([[0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000],
        [0.2000, 0.2000, 0.2000]], device='cuda:0')
a1 * b1:
tensor([[6., 6., 6., 6., 6.],
        [6., 6., 6., 6., 6.],
        [6., 6., 6., 6., 6.]], device='cuda:0')
aa.view(1, 15):
tensor([6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6.],
       device='cuda:0')
tensor([6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6., 6.],
       device='cuda:0')
tensor([[0.3857, 0.6904, 0.3359, 0.8735, 0.3501, 0.9951],
        [0.7642, 0.7241, 0.5957, 0.5391, 0.1924, 0.2412],
        [0.1162, 0.5952, 0.2729, 0.2915, 0.0693, 0.4849]], dtype=torch.float16)
tensor([[-1.5000e+00, -7.0312e-02, -5.0781e-02,  3.1250e-01, -6.0000e+01,
          3.2000e+01,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+0

In [1]:
#张量乘法

# 逐元素乘法（Element-wise multiplication）：
# * or torch.mul()()`对应位置的元素相乘，输入张量形状必须相同或可广播。

import torch

a = torch.tensor([[1, 2], [3, 4]])
b = torch.tensor([[2, 3], [4, 5]])

result = a * b
print(f"a*b result:\n {result}")

mulResult = torch.mul(a, b)
print(f"a nul b result:\n {mulResult}")

print("可以看到上述＊或mul，结果一样")

a*b result:
 tensor([[ 2,  6],
        [12, 20]])
a nul b result:
 tensor([[ 2,  6],
        [12, 20]])
可以看到上述＊或mul，结果一样


In [3]:
import torch

# 在进行 GPU 矩阵乘法（如 @ 或 torch.matmul）时，torch.float32 是最通用且兼容性最好的选择。
# 如果你的显卡支持 Tensor Cores（如 NVIDIA Ampere/Hopper 或 AMD CDNA/RDNA3），也可以尝试使用 .half() (float16) 或 .bfloat16() 来获得更高的计算吞吐量。

# 矩阵乘法：@ or torch.mm() or torch.matmul()两个矩阵相乘，第一个矩阵的列数必须等于第二个矩阵的行数。
a = torch.tensor([[1, 2], 
                  [3, 4],
                  [5, 6],
                  ], dtype=torch.float16).to("cuda")
b = torch.tensor([[-1, 0],
                  [0, 1]], dtype=torch.float16).to("cuda")

print(a @ b)

result = torch.matmul(a, b)
print(result)

print("可以看到上述@或matmul，结果一样")


# a = torch.randn(4, 3, 2, 5) # 假设a是一个4D张量，其形状为(4, 3, 2, 5)
# b = torch.randn(3, 2, 5) # 假设b是一个3D张量，其形状为(3, 2, 5)
# result = torch.matmul(a, b)
# print(result.size()) # 输出最终结果的尺寸

tensor([[-1.,  2.],
        [-3.,  4.],
        [-5.,  6.]], device='cuda:0', dtype=torch.float16)
tensor([[-1.,  2.],
        [-3.,  4.],
        [-5.,  6.]], device='cuda:0', dtype=torch.float16)
可以看到上述@或matmul，结果一样


In [6]:
import torch

A = torch.tensor([1, 2, 3], dtype=torch.float16).to("cuda")
B = torch.tensor([4, 5, 6], dtype=torch.float16).to("cuda")

# 代数定义
# 两个向量a = [a1, a2,…, an]和b = [b1, b2,…, bn]的点积定义为：
# a·b = a1b1 + a2b2 +……+ anbn。
# 仅用于两个严格的一维向量，且长度必须相同。
result = torch.dot(A, B).to("cuda")
print("Shape: %s, Value: %s" %(result.shape, result))


Shape: torch.Size([]), Value: tensor(32., device='cuda:0', dtype=torch.float16)


In [5]:
# 矩阵乘法
A = torch.tensor([[1., 2.], 
                  [3., 4.]], dtype=torch.float16).to("cuda")
B = torch.tensor([[5., 6.], 
                  [7., 8.]], dtype=torch.float16).to("cuda")

# 'ij,jk->ik'
# i: A的行, j: A的列(也是B的行), k: B的列
# j 在输出中消失 -> 对 j 求和
result = torch.einsum('ij,jk->ik', A, B)
print(result)



# 假设我们有 32 个样本，每个样本包含 10 个 5x5 的矩阵
Batch_A = torch.randn(3, 10, 5)
Batch_B = torch.randn(3, 5, 8)

print(Batch_A)
print(Batch_B)

# 'bij,bjk->bik'
# b: batch维度 (保留)
# i: A的行 (保留)
# j: 中间维度 (求和消除)
# k: B的列 (保留)
result = torch.einsum('bij,bjk->bik', Batch_A, Batch_B)

print(result) # torch.Size([3, 10, 8])

tensor([[19., 22.],
        [43., 50.]], device='cuda:0', dtype=torch.float16)
tensor([[[ 1.7873, -1.5023,  1.2423, -0.2006, -0.5865],
         [ 0.8841, -0.6305, -0.5307, -2.2733,  1.1014],
         [-0.5472, -0.3322, -0.1348,  0.7360, -1.0465],
         [ 0.0416, -0.5225,  1.1614, -0.7975,  0.5422],
         [ 1.4889,  0.6610,  0.8811,  0.5667, -0.2634],
         [-0.7452, -1.4297, -0.8689, -1.0128,  0.6431],
         [ 0.1341,  0.7196, -0.4657,  0.1623,  0.1567],
         [ 0.1686,  0.9358,  0.4637, -0.2634, -1.6047],
         [-0.8280, -0.1600, -1.4657, -0.8103, -0.8176],
         [ 1.0539,  0.8069, -0.0994,  1.1692, -0.7091]],

        [[-1.1424, -0.1291, -0.8030,  0.5256,  0.4864],
         [ 0.9024, -1.2074, -0.1863, -0.7504, -2.2562],
         [-0.5910,  1.7215, -0.9319, -1.6645,  1.7259],
         [-1.1118,  0.6771,  1.0165, -0.1003, -1.6169],
         [ 0.8411, -2.3589,  0.9675, -1.0772, -0.1209],
         [ 0.6538, -0.4150, -0.1611, -0.6440, -0.4374],
         [ 0.7132,  0.0

In [7]:
from torchvision import models
models.list_models()

['alexnet',
 'convnext_base',
 'convnext_large',
 'convnext_small',
 'convnext_tiny',
 'deeplabv3_mobilenet_v3_large',
 'deeplabv3_resnet101',
 'deeplabv3_resnet50',
 'densenet121',
 'densenet161',
 'densenet169',
 'densenet201',
 'efficientnet_b0',
 'efficientnet_b1',
 'efficientnet_b2',
 'efficientnet_b3',
 'efficientnet_b4',
 'efficientnet_b5',
 'efficientnet_b6',
 'efficientnet_b7',
 'efficientnet_v2_l',
 'efficientnet_v2_m',
 'efficientnet_v2_s',
 'fasterrcnn_mobilenet_v3_large_320_fpn',
 'fasterrcnn_mobilenet_v3_large_fpn',
 'fasterrcnn_resnet50_fpn',
 'fasterrcnn_resnet50_fpn_v2',
 'fcn_resnet101',
 'fcn_resnet50',
 'fcos_resnet50_fpn',
 'googlenet',
 'inception_v3',
 'keypointrcnn_resnet50_fpn',
 'lraspp_mobilenet_v3_large',
 'maskrcnn_resnet50_fpn',
 'maskrcnn_resnet50_fpn_v2',
 'maxvit_t',
 'mc3_18',
 'mnasnet0_5',
 'mnasnet0_75',
 'mnasnet1_0',
 'mnasnet1_3',
 'mobilenet_v2',
 'mobilenet_v3_large',
 'mobilenet_v3_small',
 'mvit_v1_b',
 'mvit_v2_s',
 'quantized_googlenet',
 '

In [11]:
alexnet = models.AlexNet()
resnet = models.resnet101(pretrained=True)
resnet

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [7]:
from torchvision import transforms
preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )])

from PIL import Image
img = Image.open("../data/p1ch2/bobby.jpg")